# 강원도 산불 예측 / 확산 위험도 모델 워크스루

이 노트북은 두 CSV를 눈으로 확인하면서 모델이 만들어지는 과정을 단계별로 따라갈 수 있게 구성했습니다.

- 산불 발생 예측: 관측소-일 단위로 산불 발생 여부를 예측합니다.
- 산불 확산 위험도 예측: 산불이 발생한 사건 중 피해면적이 1ha 이상인지 예측합니다.
- 최종 위험 점수: `산불 발생 확률 * 산불 발생 시 1ha 이상 확산 확률`로 계산합니다.

주의: 현재 모델은 제공된 두 CSV만 사용한 baseline입니다. 실제 운영 예측에는 예보 기상, 산림/지형/연료, 위치 정밀도 개선이 필요합니다.

## 1. 환경과 파일 경로 설정

작업 폴더와 원본 CSV 경로를 지정합니다. 다른 파일을 쓰려면 아래 경로만 바꾸면 됩니다.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'scripts' / 'train_models.py').exists():
            return candidate
    raise FileNotFoundError('scripts/train_models.py를 찾지 못했습니다. 노트북을 프로젝트 안에서 실행하세요.')

PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

FIRE_CSV = Path(r'C:\Users\minwoo\Desktop\data_code\cache\gangwon_forest_fire_raw_cache.csv')
WEATHER_CSV = Path(r'C:\Users\minwoo\Desktop\data_code\cache\gangwon_weather_station_all_cache.csv')
SPLIT_DATE = pd.Timestamp('2024-01-01')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('FIRE_CSV exists:', FIRE_CSV.exists(), FIRE_CSV)
print('WEATHER_CSV exists:', WEATHER_CSV.exists(), WEATHER_CSV)

PROJECT_ROOT: C:\Users\minwoo\Documents\산불 위험도 확인
FIRE_CSV exists: True C:\Users\minwoo\Desktop\data_code\cache\gangwon_forest_fire_raw_cache.csv
WEATHER_CSV exists: True C:\Users\minwoo\Desktop\data_code\cache\gangwon_weather_station_all_cache.csv


## 2. 원본 데이터 로드와 첫 확인

행/열 수, 컬럼명, 앞부분 데이터를 확인합니다.

In [2]:
fire_raw = pd.read_csv(FIRE_CSV)
weather_raw = pd.read_csv(WEATHER_CSV)

print('산불 데이터 shape:', fire_raw.shape)
display(fire_raw.head())

print('기상 데이터 shape:', weather_raw.shape)
display(weather_raw.head())

산불 데이터 shape: (1027, 16)
기상 데이터 shape: (44961, 14)


,damagearea,endday,endmonth,endtime,endyear,firecause,locbunji,locdong,locgungu,locmenu,locsi,startday,startdayofweek,startmonth,starttime,startyear
0,0.18,NaN,NaN,NaN,NaN,원인 미상,NaN,NaN,홍천,내촌,강원,19,화요일,5,NaN,2026
1,0.02,NaN,NaN,NaN,NaN,원인 미상,NaN,NaN,평창,대관령,강원,9,토요일,5,NaN,2026
2,0.02,NaN,NaN,NaN,NaN,원인 미상,NaN,NaN,홍천,남,강원,8,금요일,5,NaN,2026
3,0.05,NaN,NaN,NaN,NaN,기타,NaN,NaN,원주,귀래,강원,5,화요일,5,NaN,2026
4,0.50,NaN,NaN,NaN,NaN,원인 미상,NaN,NaN,홍천,내촌,강원,2,토요일,5,NaN,2026


,date,station,temperature,max_temperature,humidity,min_humidity,wind_speed,max_wind_speed,instant_wind_speed,rainfall,sunshine_duration,solar_radiation,ground_temperature,station_name
0,2011-01-01,101,-9.0,-0.8,74.0,42.0,0.2,1.1,2.1,0.0,7.4,9.72,-2.6,춘천
1,2011-01-02,101,-8.3,0.0,84.3,43.0,0.3,1.4,2.4,0.0,4.2,7.02,-2.6,춘천
2,2011-01-03,101,-7.5,0.5,79.5,44.0,0.2,1.1,1.7,0.0,6.3,8.14,-2.9,춘천
3,2011-01-04,101,-5.9,0.9,77.5,44.0,0.7,2.7,5.0,0.0,5.3,7.63,-2.3,춘천
4,2011-01-05,101,-3.9,0.1,65.3,47.0,2.0,4.4,8.1,0.0,6.3,9.65,-1.3,춘천


## 3. 결측치와 기본 분포 확인

산불 피해면적, 발생 지역, 기상 관측소 범위를 확인합니다.

In [3]:
print('산불 컬럼별 결측률')
display(fire_raw.isna().mean().sort_values(ascending=False).to_frame('missing_rate'))

print('피해면적 요약')
display(fire_raw['damagearea'].describe().to_frame())

print('시군구별 산불 건수')
display(fire_raw['locgungu'].value_counts().to_frame('fire_count'))

print('기상 관측소별 행 수')
display(weather_raw['station_name'].value_counts().sort_index().to_frame('weather_rows'))

산불 컬럼별 결측률
피해면적 요약
시군구별 산불 건수
기상 관측소별 행 수


,missing_rate
endday,1.000000
endmonth,1.000000
endtime,1.000000
endyear,1.000000
locdong,1.000000
locbunji,1.000000
starttime,1.000000
locmenu,0.025316
firecause,0.000000
damagearea,0.000000


,damagearea
count,1027.000000
mean,11.104041
std,147.123369
min,0.010000
25%,0.050000
50%,0.120000
75%,0.500000
max,4190.380000


,fire_count
locgungu,
홍천,134
춘천,130
강릉,87
원주,81
횡성,75
삼척,68
평창,57
인제,54
영월,52


,weather_rows
station_name,
강릉,5621
속초,5621
원주,5621
인제,5620
정선군,5619
춘천,5621
태백,5621
홍천,5617


## 4. 모델링 함수 불러오기

`scripts/train_models.py`에 있는 전처리, 학습, 평가 함수를 재사용합니다. 노트북에서는 각 중간 결과를 바로 확인합니다.

In [4]:
from train_models import (
    LOCATION_TO_STATION,
    LARGE_FIRE_THRESHOLD_HA,
    add_weather_features,
    build_occurrence_dataset,
    build_spread_dataset,
    classification_metrics,
    feature_signal_table,
    load_fire_events,
    predict_with_model,
    risk_band,
    train_binary_model,
)

print('확산 기준 피해면적 ha:', LARGE_FIRE_THRESHOLD_HA)
display(pd.Series(LOCATION_TO_STATION, name='mapped_station').to_frame())

확산 기준 피해면적 ha: 1.0


,mapped_station
강릉,강릉
고성,속초
동해,강릉
삼척,태백
속초,속초
양구,인제
양양,속초
영월,정선군
원주,원주
인제,인제


## 5. 날짜/관측소 기준 전처리

산불 데이터는 발생 날짜와 대표 관측소를 만들고, 기상 데이터에는 3일/7일 강수량, 건조일수, 계절성 변수를 추가합니다.

In [5]:
fire = load_fire_events(FIRE_CSV)
weather = add_weather_features(weather_raw)

print('전처리 후 산불:', fire.shape, fire['date'].min().date(), '~', fire['date'].max().date())
display(fire[['date', 'locgungu', 'station_name', 'damagearea', 'large_fire', 'firecause']].head(10))

print('전처리 후 기상:', weather.shape, weather['date'].min().date(), '~', weather['date'].max().date())
display(weather[['date', 'station_name', 'temperature', 'humidity', 'rainfall', 'rainfall_7d_sum', 'dry_days']].head(10))

전처리 후 산불: (1027, 19) 2011-01-22 ~ 2026-05-19
전처리 후 기상: (44961, 25) 2011-01-01 ~ 2026-05-22


,date,locgungu,station_name,damagearea,large_fire,firecause
0,2026-05-19,홍천,홍천,0.18,0,원인 미상
1,2026-05-09,평창,정선군,0.02,0,원인 미상
2,2026-05-08,홍천,홍천,0.02,0,원인 미상
3,2026-05-05,원주,원주,0.05,0,기타
4,2026-05-02,홍천,홍천,0.50,0,원인 미상
5,2026-05-01,원주,원주,0.07,0,입산자 실화
6,2026-04-29,홍천,홍천,0.25,0,입산자 실화
7,2026-04-27,춘천,춘천,0.90,0,건축물 실화
8,2026-04-27,인제,인제,0.23,0,입산자 실화
9,2026-04-27,화천,춘천,0.20,0,기타


,date,station_name,temperature,humidity,rainfall,rainfall_7d_sum,dry_days
5621,2011-01-01,강릉,-1.5,71.4,6.0,6.0,0
5622,2011-01-02,강릉,0.6,66.4,0.5,6.5,1
5623,2011-01-03,강릉,-0.9,83.6,6.5,13.0,0
5624,2011-01-04,강릉,-0.1,47.1,0.0,13.0,1
5625,2011-01-05,강릉,-0.6,46.5,0.0,13.0,2
5626,2011-01-06,강릉,-3.5,45.3,0.0,13.0,3
5627,2011-01-07,강릉,-2.3,32.8,0.0,13.0,4
5628,2011-01-08,강릉,0.5,38.6,0.0,7.0,5
5629,2011-01-09,강릉,-1.3,39.1,0.0,6.5,6
5630,2011-01-10,강릉,-3.5,31.8,0.0,0.0,7


## 6. 학습 데이터 만들기

- 발생 모델 데이터: 모든 관측소-일 조합에 산불 발생 여부를 붙입니다.
- 확산 모델 데이터: 실제 산불 사건만 남겨 1ha 이상 여부를 붙입니다.

In [6]:
occurrence_df = build_occurrence_dataset(weather, fire)
spread_df = build_spread_dataset(weather, fire)

print('발생 모델 데이터:', occurrence_df.shape)
display(occurrence_df[['date', 'station_name', 'fire_count', 'fire_occurrence', 'max_damagearea']].head())
display(occurrence_df['fire_occurrence'].value_counts(normalize=True).rename('rate').to_frame())

print('확산 모델 데이터:', spread_df.shape)
display(spread_df[['date', 'station_name', 'damagearea', 'large_fire', 'temperature', 'humidity', 'max_wind_speed']].head())
display(spread_df['large_fire'].value_counts(normalize=True).rename('rate').to_frame())

발생 모델 데이터: (44961, 29)
확산 모델 데이터: (1027, 42)


,date,station_name,fire_count,fire_occurrence,max_damagearea
0,2011-01-01,강릉,0,0,0.0
1,2011-01-02,강릉,0,0,0.0
2,2011-01-03,강릉,0,0,0.0
3,2011-01-04,강릉,0,0,0.0
4,2011-01-05,강릉,0,0,0.0


,rate
fire_occurrence,
0,0.978359
1,0.021641


,date,station_name,damagearea,large_fire,temperature,humidity,max_wind_speed
0,2026-05-19,홍천,0.18,0,21.3,52.0,2.7
1,2026-05-09,정선군,0.02,0,12.5,54.8,4.3
2,2026-05-08,홍천,0.02,0,14.4,49.5,6.3
3,2026-05-05,원주,0.05,0,14.2,53.3,4.7
4,2026-05-02,홍천,0.50,0,16.4,62.8,3.9


,rate
large_fire,
0,0.848101
1,0.151899


## 7. 시간 기준 학습/검증 분리

`2024-01-01` 이전은 학습, 이후는 테스트로 사용합니다. 미래 데이터를 섞지 않기 위한 분리입니다.

In [7]:
def split_summary(df, target):
    rows = []
    for name, part in [('train', df[df['date'] < SPLIT_DATE]), ('test', df[df['date'] >= SPLIT_DATE])]:
        rows.append({
            'split': name,
            'rows': len(part),
            'positives': int(part[target].sum()),
            'positive_rate': float(part[target].mean()),
            'start': part['date'].min().date(),
            'end': part['date'].max().date(),
        })
    return pd.DataFrame(rows)

print('발생 모델 split')
display(split_summary(occurrence_df, 'fire_occurrence'))

print('확산 모델 split')
display(split_summary(spread_df, 'large_fire'))

발생 모델 split
확산 모델 split


,split,rows,positives,positive_rate,start,end
0,train,37984,852,0.022430,2011-01-01,2023-12-31
1,test,6977,121,0.017343,2024-01-01,2026-05-22


,split,rows,positives,positive_rate,start,end
0,train,903,137,0.151717,2011-01-22,2023-12-29
1,test,124,19,0.153226,2024-03-15,2026-05-19


## 8. 산불 발생 모델 학습

관측소-일 단위 산불 발생 확률을 학습합니다.

In [8]:
occurrence_model, occurrence_metrics, _ = train_binary_model(
    occurrence_df,
    'fire_occurrence',
    SPLIT_DATE,
    {
        'model_name': 'forest_fire_occurrence',
        'description': '관측소-일 단위 산불 발생 여부 예측',
    },
)

display(pd.DataFrame([occurrence_metrics['train'], occurrence_metrics['test']], index=['train', 'test']))
display(pd.DataFrame(feature_signal_table(occurrence_model, top_n=15)))

,rows,positive_rate,roc_auc,average_precision,brier,threshold,precision,recall,f1,tp,tn,fp,fn,top_10pct_positive_rate,top_10pct_lift
train,37984,0.022430,0.848977,0.135775,0.020571,0.085848,0.156316,0.348592,0.215843,297,35529,1603,555,0.116083,5.175239
test,6977,0.017343,0.850371,0.087885,0.016376,0.085848,0.101587,0.264463,0.146789,32,6573,283,89,0.090258,5.204374


,feature,coefficient,direction,abs_coefficient
0,rainfall_7d_sum,-0.605921,risk_down,0.605921
1,min_humidity_3d_avg,-0.488504,risk_down,0.488504
2,max_temperature,0.336420,risk_up,0.336420
3,rainfall_3d_sum,-0.316647,risk_down,0.316647
4,humidity,-0.290456,risk_down,0.290456
5,min_humidity,-0.276994,risk_down,0.276994
6,doy_sin,0.217607,risk_up,0.217607
7,month_sin,0.208546,risk_up,0.208546
8,station=춘천,0.203092,risk_up,0.203092
9,dry_days,0.186584,risk_up,0.186584


## 9. 산불 확산 모델 학습

실제 산불 사건 중 피해면적이 1ha 이상이 될 확률을 학습합니다.

In [9]:
spread_model, spread_metrics, _ = train_binary_model(
    spread_df,
    'large_fire',
    SPLIT_DATE,
    {
        'model_name': 'forest_fire_spread',
        'description': '산불 발생 시 1ha 이상 확산 여부 예측',
        'large_fire_threshold_ha': LARGE_FIRE_THRESHOLD_HA,
    },
)

display(pd.DataFrame([spread_metrics['train'], spread_metrics['test']], index=['train', 'test']))
display(pd.DataFrame(feature_signal_table(spread_model, top_n=15)))

,rows,positive_rate,roc_auc,average_precision,brier,threshold,precision,recall,f1,tp,tn,fp,fn,top_10pct_positive_rate,top_10pct_lift
train,903,0.151717,0.767538,0.367895,0.113226,0.172566,0.335714,0.686131,0.450839,94,580,186,43,0.417582,2.752386
test,124,0.153226,0.588221,0.180005,0.147545,0.172566,0.200000,0.473684,0.281250,9,69,36,10,0.076923,0.502024


,feature,coefficient,direction,abs_coefficient
0,instant_wind_speed,0.485414,risk_up,0.485414
1,max_temperature,0.369252,risk_up,0.369252
2,doy_cos,0.347834,risk_up,0.347834
3,month_cos,0.338935,risk_up,0.338935
4,station=강릉,-0.318605,risk_down,0.318605
5,humidity,-0.293390,risk_down,0.293390
6,station=태백,0.232645,risk_up,0.232645
7,month_sin,0.221396,risk_up,0.221396
8,dry_days,0.196167,risk_up,0.196167
9,ground_temperature,0.194387,risk_up,0.194387


## 10. 전체 날짜별 위험도 계산

발생 확률과 확산 확률을 곱해서 최종 위험 점수를 만듭니다. 위험 등급은 학습 구간 점수의 상위 20%, 상위 5% 기준으로 나눕니다.

In [10]:
risk_df = occurrence_df.copy()
risk_df['fire_probability'] = predict_with_model(risk_df, occurrence_model)
risk_df['spread_probability_if_fire'] = predict_with_model(risk_df, spread_model)
risk_df['spread_risk_score'] = risk_df['fire_probability'] * risk_df['spread_probability_if_fire']

train_scores = risk_df.loc[risk_df['date'] < SPLIT_DATE, 'spread_risk_score']
medium_threshold = float(train_scores.quantile(0.80))
high_threshold = float(train_scores.quantile(0.95))
risk_df['risk_band'] = [risk_band(score, high_threshold, medium_threshold) for score in risk_df['spread_risk_score']]

print('medium_threshold:', medium_threshold)
print('high_threshold:', high_threshold)
display(risk_df[['date', 'station_name', 'fire_probability', 'spread_probability_if_fire', 'spread_risk_score', 'risk_band']].head())
display(risk_df['risk_band'].value_counts().to_frame('rows'))

medium_threshold: 0.0032488902713279943
high_threshold: 0.015342370845249125


,date,station_name,fire_probability,spread_probability_if_fire,spread_risk_score,risk_band
0,2011-01-01,강릉,0.002958,0.022834,0.000068,low
1,2011-01-02,강릉,0.004351,0.014510,0.000063,low
2,2011-01-03,강릉,0.000989,0.008248,0.000008,low
3,2011-01-04,강릉,0.006833,0.061532,0.000420,low
4,2011-01-05,강릉,0.005799,0.098500,0.000571,low


,rows
risk_band,
low,35923
medium,6803
high,2235


## 11. 최신 날짜 관측소별 위험도 확인

가장 최근 기상일의 관측소별 예측 결과입니다.

In [11]:
latest_date = risk_df['date'].max()
latest = risk_df[risk_df['date'] == latest_date].copy()
latest_cols = [
    'date', 'station_name', 'fire_probability', 'spread_probability_if_fire',
    'spread_risk_score', 'risk_band', 'temperature', 'humidity', 'min_humidity',
    'max_wind_speed', 'rainfall', 'dry_days'
]
display(latest[latest_cols].sort_values('spread_risk_score', ascending=False))

,date,station_name,fire_probability,spread_probability_if_fire,spread_risk_score,risk_band,temperature,humidity,min_humidity,max_wind_speed,rainfall,dry_days
33722,2026-05-22,춘천,0.008491,0.019060,1.618400e-04,low,18.7,54.4,21.0,5.7,0.0,1
44960,2026-05-22,홍천,0.006420,0.014508,9.313756e-05,low,18.8,55.1,19.0,4.6,0.0,1
22482,2026-05-22,인제,0.003816,0.022801,8.701676e-05,low,16.8,61.4,24.0,4.9,0.0,1
16862,2026-05-22,원주,0.009317,0.008907,8.299359e-05,low,19.7,56.4,20.0,4.0,0.0,1
28101,2026-05-22,정선군,0.002235,0.012391,2.769130e-05,low,15.6,62.9,30.0,5.9,0.0,1
39343,2026-05-22,태백,0.000195,0.005398,1.050837e-06,low,12.6,78.0,45.0,4.9,1.0,0
11241,2026-05-22,속초,0.000178,0.002019,3.588910e-07,low,15.7,81.4,73.0,5.1,0.0,1
5620,2026-05-22,강릉,0.000217,0.000782,1.693576e-07,low,16.6,80.3,56.0,4.4,0.0,1


## 12. 결과 파일 저장

노트북에서 계산한 결과를 별도 CSV/JSON으로 저장합니다.

In [12]:
notebook_output_dir = PROJECT_ROOT / 'outputs' / 'notebook'
notebook_output_dir.mkdir(parents=True, exist_ok=True)

latest[latest_cols].sort_values('spread_risk_score', ascending=False).to_csv(
    notebook_output_dir / 'latest_risk_from_notebook.csv',
    index=False,
    encoding='utf-8-sig',
)

metrics = {
    'occurrence': occurrence_metrics,
    'spread': spread_metrics,
    'risk_band_thresholds': {
        'medium_min': medium_threshold,
        'high_min': high_threshold,
    },
}
(notebook_output_dir / 'metrics_from_notebook.json').write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('저장 완료:', notebook_output_dir)

저장 완료: C:\Users\minwoo\Documents\산불 위험도 확인\outputs\notebook


## 13. 해석 요약

- 발생 모델은 희소 이벤트 문제이므로 정확도보다 ROC AUC, average precision, 상위 위험군 lift를 봅니다.
- 확산 모델은 실제 산불 사건 수가 적고 1ha 이상 사례가 더 적어 변동성이 큽니다.
- `spread_risk_score`는 절대 확률이라기보다 관측소/날짜 간 상대 위험도 비교에 더 적합합니다.
- 실제 예측에는 관측값 대신 예보 기상값을 같은 컬럼 구조로 넣어야 합니다.